# Aurum Market | Práctica de Bases de Datos Vectoriales

**Motor de descubrimiento y control de catálogo**

Este notebook documenta una solución operativa para:

1. Recuperación semántica de productos por intención.
2. Filtrado condicional por metadatos (`brand`).
3. Control de altas potencialmente duplicadas.

## 1. Objetivos y enfoque

Pasos del notebook:

- Definir el contrato de datos y la interfaz de resultados.
- Construir un baseline léxico interpretable.
- Comparar representaciones textuales relevantes.
- Implementar un índice/vector store local y persistente.
- Exponer búsquedas globales y filtradas.
- Aplicar eventos de catálogo de forma idempotente.
- Diseñar y calibrar la regla de duplicados.
- Evaluar con métricas reproducibles sobre desarrollo.

## 2. Requisitos y entorno

Este notebook requiere un entorno Python que incluya:

- `pandas`
- `numpy`
- `scikit-learn`
- `sentence-transformers` (para embeddings)
- `faiss-cpu` (opcionalmente, para indexación ANN)

Si el entorno no dispone de `sentence-transformers` o `faiss-cpu`, el notebook ofrece alternativas para seguir el flujo con una implementación de respaldo.

In [2]:
import sys
print(f"Python ejecutable: {sys.executable}")
print(f"Versión: {sys.version}")
print(f"Ruta de búsqueda: {sys.path}")

Python ejecutable: c:\Users\diego\AppData\Local\Python\pythoncore-3.14-64\python.exe
Versión: 3.14.3 (tags/v3.14.3:323c59a, Feb  3 2026, 16:04:56) [MSC v.1944 64 bit (AMD64)]
Ruta de búsqueda: ['c:\\Users\\diego\\AppData\\Local\\Python\\pythoncore-3.14-64\\python314.zip', 'c:\\Users\\diego\\AppData\\Local\\Python\\pythoncore-3.14-64\\DLLs', 'c:\\Users\\diego\\AppData\\Local\\Python\\pythoncore-3.14-64\\Lib', 'c:\\Users\\diego\\AppData\\Local\\Python\\pythoncore-3.14-64', '', 'c:\\Users\\diego\\AppData\\Local\\Python\\pythoncore-3.14-64\\Lib\\site-packages']


In [6]:
import sys
!{sys.executable} -m pip install sentence-transformers torch faiss-cpu

   ---------------------------------------- 0.0/611.3 kB ? eta -:--:--
   ---------------------------------------- 611.3/611.3 kB 25.6 MB/s  0:00:00
   ---------------------------------------- 0.0/11.6 MB ? eta -:--:--
   ----------------------------- ---------- 8.7/11.6 MB 41.3 MB/s eta 0:00:01
   ---------------------------------------- 11.6/11.6 MB 37.9 MB/s  0:00:00
   ---------------------------------------- 0.0/780.8 kB ? eta -:--:--
   ---------------------------------------- 780.8/780.8 kB 30.5 MB/s  0:00:00
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------------------------------------- 4.0/4.0 MB 40.5 MB/s  0:00:00
   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   ---------------------------------------- 2.7/2.7 MB 40.8 MB/s  0:00:00
   ---------------------------------------- 0.0/122.1 MB ? eta -:--:--
   -- ------------------------------------- 8.7/122.1 MB 42.5 MB/s eta 0:00:03
   ----- ---------------------------------


[notice] A new release of pip is available: 26.1.1 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
from pathlib import Path
import json
import math
import time
from dataclasses import dataclass
from typing import Optional, List, Dict, Tuple

import numpy as np
import pandas as pd

try:
    from sentence_transformers import SentenceTransformer
    EMBEDDING_AVAILABLE = True
except Exception:
    EMBEDDING_AVAILABLE = False

try:
    import faiss
    FAISS_AVAILABLE = True
except Exception:
    FAISS_AVAILABLE = False

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Proyecto: {PROJECT_ROOT}")
print(f"Embedding model disponible: {EMBEDDING_AVAILABLE}")
print(f"Resultados guardados en: {RESULTS_DIR}")
print(f"Faiss disponible: {FAISS_AVAILABLE}")

if not EMBEDDING_AVAILABLE:
    print("\nPara habilitar los embeddings, instala en el entorno activo:")
    print("pip install sentence-transformers torch")

if not FAISS_AVAILABLE:
    print("\nPara habilitar FAISS, instala en el entorno activo:")
    print("pip install faiss-cpu")

Proyecto: c:\Users\diego\Documents\PontIA\BDVextoriales\Prueba-Practica
Embedding model disponible: True
Resultados guardados en: c:\Users\diego\Documents\PontIA\BDVextoriales\Prueba-Practica\results
Faiss disponible: True


## 3. Carga de datos y esquema

El notebook trabaja con los ficheros definidos. La muestra (`catalogo_muestra.csv`) sirve para desarrollo y el resto de ficheros para construir las pruebas del caso.

In [9]:
catalog_sample = pd.read_csv(DATA_DIR / "catalogo_muestra.csv")
#catalog_full = pd.read_csv(DATA_DIR / "catalogo_productos.csv")#Cargar catalogo_productos.csv para ejecutar sobre la muestra completa. 
queries_dev = pd.read_csv(DATA_DIR / "consultas_desarrollo.csv")
queries_eval = pd.read_csv(DATA_DIR / "consultas_evaluacion.csv")
queries_filtered = pd.read_csv(DATA_DIR / "consultas_filtradas.csv")
relevances_dev = pd.read_csv(DATA_DIR / "relevancias_desarrollo.csv")
duplicate_dev = pd.read_csv(DATA_DIR / "altas_desarrollo.csv")
events_catalog = pd.read_csv(DATA_DIR / "eventos_catalogo.csv")

print("catalogo_muestra:", catalog_sample.shape)
print("consultas_desarrollo:", queries_dev.shape)
print("consultas_evaluacion:", queries_eval.shape)
print("consultas_filtradas:", queries_filtered.shape)
print("relevancias_desarrollo:", relevances_dev.shape)
print("altas_desarrollo", duplicate_dev.shape)
print("eventos_catalogo:", events_catalog.shape)

catalogo_muestra: (1500, 9)
consultas_desarrollo: (8, 4)
consultas_evaluacion: (12, 3)
consultas_filtradas: (4, 6)
relevancias_desarrollo: (248, 4)
altas_desarrollo (14, 7)
eventos_catalogo: (24, 12)


In [10]:
queries_eval[
    ["evaluation_id", "query_text", "query_type"]
].style.set_properties(subset=["query_text"], **{"font-weight": "bold"})


,evaluation_id,query_text,query_type
0,EVAL-100455-context,taladro sin cable de 24 voltios que venga con su batería,context
1,EVAL-100455-direct,taladro 24v batería,direct
2,EVAL-100455-semantic,quiero una herramienta inalámbrica potente para perforar sin depender de un enchufe,semantic
3,EVAL-101352-context,"tele de tamaño reducido para una cocina, alrededor de 28 pulgadas",context
4,EVAL-101352-direct,television 28 pulgadas,direct
5,EVAL-101352-semantic,busco un televisor pequeño de unas setenta centímetros para la cocina,semantic
6,EVAL-93437-context,me duele la espalda al trabajar y necesito una silla con buen apoyo lumbar,context
7,EVAL-93437-direct,sillas oficina ergonomicas,direct
8,EVAL-93437-semantic,necesito un asiento cómodo para trabajar ocho horas con buen apoyo para la espalda,semantic
9,EVAL-96202-context,pieza para sujetar un aire acondicionado en el hueco de la ventana,context


### 3.1 Esquema del catálogo

Los campos principales son:

- `record_id`: UUID estable que usaremos como `id` vectorial.
- `product_id`: identificador comercial del producto.
- `title`, `brand`, `color`, `text`: información textual y metadatos.
- `catalog_version`, `active`: versión y visibilidad.

El sistema debe tratar valores ausentes de forma consistente. En este notebook, los valores void o null seran ignorados.

In [11]:
print(catalog_sample.columns.tolist())
print(catalog_sample.dtypes)
print("Nulos por columna:\n", catalog_sample.isna().sum())

['record_id', 'product_id', 'title', 'brand', 'color', 'locale', 'text', 'catalog_version', 'active']
record_id            str
product_id           str
title                str
brand                str
color                str
locale               str
text                 str
catalog_version    int64
active              bool
dtype: object
Nulos por columna:
 record_id            0
product_id           0
title                0
brand               44
color              549
locale               0
text                 0
catalog_version      0
active               0
dtype: int64


## 4. Contrato de relevancia

Convertimos las etiquetas ESCI a relevancia graduada:

- `E` = 3
- `S` = 2
- `C` = 1
- `I` = 0

Esta transformación permite calcular nDCG y mantener la información sobre la severidad de los errores de ranking.

In [12]:
import plotly.express as px


def ndcg_at_k(ranked_relevances, k=10):
    ranked_relevances = ranked_relevances[:k]
    dcg = sum((2**rel - 1) / math.log2(idx + 2) for idx, rel in enumerate(ranked_relevances))
    ideal = sorted(ranked_relevances, reverse=True)
    idcg = sum((2**rel - 1) / math.log2(idx + 2) for idx, rel in enumerate(ideal))
    return dcg / idcg if idcg > 0 else 0.0


def recall_at_k(ranked_ids, relevant_ids, k=10):
    if not relevant_ids:
        return 0.0
    return sum(1 for pid in ranked_ids[:k] if pid in relevant_ids) / len(relevant_ids)


def mrr_at_k(ranked_ids, relevant_ids, k=10):
    for idx, pid in enumerate(ranked_ids[:k], start=1):
        if pid in relevant_ids:
            return 1.0 / idx
    return 0.0


def qrels_by_query(qrels):
    return {
        str(q): set(group["product_id"].astype(str).tolist())
        for q, group in qrels.groupby("query_id")
    }

qrels_by_query_dev = qrels_by_query(relevances_dev)

label_counts = (
    relevances_dev["relevance"]
    .map({3: "E", 2: "S", 1: "C", 0: "I"})
    .value_counts()
    .reset_index()
)
label_counts.columns = ["label", "pairs"]
label_counts["meaning"] = label_counts["label"].map({
    "E": "Exacta",
    "S": "Sensible",
    "C": "Compatible",
    "I": "Irrelevante",
})

label_figure = px.bar(
    label_counts,
    x="label",
    y="pairs",
    color="meaning",
    text="pairs",
    color_discrete_sequence=px.colors.qualitative.Safe,
)
label_figure.update_layout(
    title="Distribución de relevancia graduada en la muestra",
    xaxis_title="Etiqueta ESCI",
    yaxis_title="Pares consulta-producto",
)
label_figure.show()

## 5. Baseline léxico

Construimos un baseline TF-IDF sobre `title` + `text`.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

catalog_text = (catalog_sample["title"].fillna("") + " " + catalog_sample["text"].fillna(""))
vectorizer = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), stop_words=None)
X_tfidf = vectorizer.fit_transform(catalog_text)
print("TF-IDF matrix shape:", X_tfidf.shape)


def search_lexical(query, top_k=10):
    q_vec = vectorizer.transform([query])
    scores = linear_kernel(q_vec, X_tfidf).flatten()
    top_idx = np.argsort(scores)[::-1][:top_k]
    return [catalog_sample.iloc[i]["product_id"] for i in top_idx], scores[top_idx].tolist()

search_lexical("zapatillas cómodas para correr", top_k=5)

TF-IDF matrix shape: (1500, 20000)


(['B08YVLVKFD', 'B095PN46NB', 'B07C7ST5JD', 'B08SMSZPNC', 'B07F2JVRZT'],
 [0.417074852127889,
  0.3043937668243131,
  0.13487094463873006,
  0.11437782804234377,
  0.10920963138597688])

In [19]:
def evaluate_search(search_fn, queries, qrels, top_k=10):
    results = []
    qrels_map = qrels_by_query(qrels)
    for _, row in queries.iterrows():
        qid = str(row["query_id"])
        ranked_ids, _ = search_fn(row["query_text"], top_k=top_k)
        relevant = qrels_map.get(qid, set())
        ranked_rels = [relevances_dev.loc[(relevances_dev["query_id"] == int(qid)) & (relevances_dev["product_id"] == pid), "relevance"].iloc[0] if pid in relevant else 0 for pid in ranked_ids]
        results.append((int(qid), ndcg_at_k(ranked_rels, top_k), recall_at_k(ranked_ids, relevant, top_k), mrr_at_k(ranked_ids, relevant, top_k)))
    return pd.DataFrame(results, columns=["query_id", "ndcg", "recall", "mrr"])

lexical_results = evaluate_search(search_lexical, queries_dev, relevances_dev)
lexical_results[["ndcg", "recall", "mrr"]].mean()

ndcg      0.817263
recall    0.201563
mrr       0.892857
dtype: float64

## 6. Representación densa

Definimos el texto de entrada para el embedding combinando título, marca, color y descripción. Esta selección intenta preservar tanto la identificación del producto como los atributos útiles para catalogar.


In [14]:
def build_product_text(row):
    parts = []
    if pd.notna(row.get("title")) and str(row.get("title")).strip():
        parts.append(str(row["title"]).strip())
    if pd.notna(row.get("brand")) and str(row.get("brand")).strip():
        parts.append(f"Marca: {row['brand']}")
    if pd.notna(row.get("color")) and str(row.get("color")).strip():
        parts.append(f"Color: {row['color']}")
    if pd.notna(row.get("text")) and str(row.get("text")).strip():
        parts.append(str(row["text"]).strip())
    return " | ".join(parts)

catalog_sample["text_for_embedding"] = catalog_sample.apply(build_product_text, axis=1)
catalog_sample[["product_id", "text_for_embedding"]].head(3)

,product_id,text_for_embedding
0,B0818K237B,Kanlin1986 Vestido Largo De Navidad para Mujer...
1,B086YX9RK5,IQOS Kit Iqos 3 Duo Blue Opk 1 200 g | Marca: ...
2,B07FRXCFJ1,ELINKUME Lámpara de pie regulable LED Lámpara ...


In [15]:
if EMBEDDING_AVAILABLE:
    model_name = "all-MiniLM-L6-v2"
    model = SentenceTransformer(model_name)
    embeddings = model.encode(catalog_sample["text_for_embedding"].tolist(), convert_to_numpy=True, show_progress_bar=True)
    print("Embeddings shape:", embeddings.shape)
else:
    from sklearn.feature_extraction.text import TfidfVectorizer
    # En el fallback local usamos stop_words=None para compatibilidad con la versión de sklearn instalada.
    emb_vectorizer = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), stop_words=None)
    embeddings = emb_vectorizer.fit_transform(catalog_sample["text_for_embedding"]).toarray()
    print("Fallback embeddings shape:", embeddings.shape)

embeddings = embeddings.astype(np.float32)
row_norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
embeddings = embeddings / np.maximum(row_norms, 1e-10)

c:\Users\diego\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\diego\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Batches: 100%|██████████| 47/47 [00:26<00:00,  1.80it/s]

c:\Users\diego\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\diego\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Batches: 100%|██████████| 47/47 [00:26<00:00,  1.80it/s]

Embeddings shape: (1500, 384)


### 6.1 Métrica y semántica del score

Usamos similitud coseno como métrica. Con embeddings normalizados, un score mayor es mayor similitud semántica.

In [20]:
import plotly.graph_objects as go

if FAISS_AVAILABLE:
    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings)
    print("Índice Faiss construido con", index.ntotal, "vectores")
else:
    index = None
    print("Faiss no está disponible. Usaremos búsqueda exhaustiva con NumPy.")

if EMBEDDING_AVAILABLE:
    def encode_query(query_text):
        q_vec = model.encode([query_text], convert_to_numpy=True)
        q_vec = q_vec.astype(np.float32)
        q_norm = np.linalg.norm(q_vec, axis=1, keepdims=True)
        return q_vec / np.maximum(q_norm, 1e-10)
else:
    def encode_query(query_text):
        q_vec = emb_vectorizer.transform([query_text]).toarray().astype(np.float32)
        q_norm = np.linalg.norm(q_vec, axis=1, keepdims=True)
        return q_vec / np.maximum(q_norm, 1e-10)

query_text = "zapatillas cómodas para correr"
query_vector = encode_query(query_text).reshape(-1)

vector_names = [
    "zapatillas running comfort",
    "zapatillas deportivas mujer",
    "botines de trail",
    "chanclas deportivas",
    "sudadera para running",
]
product_vectors = np.array(
    [[0.95, 0.92], [0.90, 0.62], [0.82, 0.15], [0.38, 0.88], [0.12, 0.74]],
    dtype=np.float32,
)
query_vector_example = np.array([0.92, 0.90], dtype=np.float32)

vector_figure = go.Figure()
for product_name, product_vector in zip(vector_names, product_vectors, strict=True):
    vector_figure.add_trace(
        go.Scatter(
            x=[0, product_vector[0]],
            y=[0, product_vector[1]],
            name=product_name,
            mode="lines+markers",
            marker={"size": 8},
            line={"width": 2},
        )
    )

vector_figure.add_trace(
    go.Scatter(
        x=[0, query_vector_example[0]],
        y=[0, query_vector_example[1]],
        name="consulta",
        mode="lines+markers",
        marker={"size": 10, "color": "#ef4444"},
        line={"width": 6, "color": "#ef4444"},
    )
)
vector_figure.update_layout(
    title="Consulta y candidatos en un espacio de dos dimensiones",
    xaxis_title="Afinidad con 'deporte / running'",
    yaxis_title="Afinidad con 'comodidad / estilo'",
    height=520,
    template="plotly_white",
)
vector_figure.show()


def search_dense(query_text, top_k=10, filter_brand: Optional[str] = None):
    q_vec = encode_query(query_text)
    if index is not None:
        scores, indices = index.search(q_vec, top_k + 20)
        scores, indices = scores[0], indices[0]
    else:
        scores = embeddings.dot(q_vec.flatten())
        indices = np.argsort(scores)[::-1]
        scores = scores[indices][: top_k + 20]
        indices = indices[: top_k + 20]

    results = []
    for score, idx in zip(scores, indices):
        row = catalog_sample.iloc[idx]
        if filter_brand is not None:
            if str(row["brand"]).strip().lower() != str(filter_brand).strip().lower():
                continue
        results.append({
            "product_id": row["product_id"],
            "title": row["title"],
            "brand": row["brand"],
            "score": float(score),
            "record_id": row["record_id"],
        })
        if len(results) >= top_k:
            break
    return results

search_dense("zapatillas cómodas para correr", top_k=5)

Índice Faiss construido con 1500 vectores


[{'product_id': 'B08ZXMXS8J',
  'title': 'Zapatillas Casual Hombre Running Zapatos Moda Sneakers Deportivas Gimnasio',
  'brand': 'SANNAX',
  'score': 0.5219706296920776,
  'record_id': '934d64df-7895-5e72-9114-c3ffc274030c'},
 {'product_id': 'B07FFL8SFV',
  'title': 'Zapatillas para Hombre Jazz Original de Saucony, Color Azul, Talla 41 EU',
  'brand': 'Saucony',
  'score': 0.46265172958374023,
  'record_id': '712a3ef5-810a-5c04-8d09-b90b49c83c88'},
 {'product_id': 'B08Q87NDBT',
  'title': 'ISAKEN Gato Animal de Peluche Gato Juguetes de Peluche, Muñeca de Felpa Almohada para Abrazar El Sofá del Hogar Cojín Trasero para Niños Regalo',
  'brand': 'ISAKEN',
  'score': 0.45498794317245483,
  'record_id': '2b7254a7-9a76-57af-a647-55fc2db63ba5'},
 {'product_id': 'B079GWS3ZX',
  'title': 'Zapatillas CTAS Lift Hi Platform Blanco, Mujer',
  'brand': 'Converse',
  'score': 0.4439670443534851,
  'record_id': '469d9042-cf67-5534-b892-57cd52c95319'},
 {'product_id': 'B079RVH8H4',
  'title': '2 Parc

### 6.2 Índice ANN con HNSW y semántica del score

La clase de sesión de prácticas insiste en separar dos capas: la representación del texto y la estructura que permite recuperar vecinos a escala. El notebook base ya usa un índice exacto sobre embeddings normalizados; aquí añadimos la versión ANN (HNSW) y la misma disciplina de diseño que vemos en los notebooks de clase:

- `IndexFlatIP` sirve como oráculo para medir la calidad de la representación.
- `IndexHNSWFlat` permite un acceso aproximado con coste proporcional a los vecinos cercanos.
- El filtrado de marca se mantiene como restricción de consulta, no como corrección del ranking.
- La semántica del score se conserva: como los vectores están normalizados, un score más alto representa mayor similitud coseno.

Esto ayuda a distinguir si un error viene del modelo, del índice o del filtro.

In [ ]:
from dataclasses import dataclass

@dataclass
class SearchHit:
    record_id: str
    product_id: str
    title: str
    brand: str
    score: float
    score_kind: str = "similarity"
    higher_is_better: bool = True
    rank: int = 0


def build_hnsw_index(embeddings, *, M=24, ef_construction=120, ef_search=128):
    """Construye un índice ANN basado en HNSW para similitud coseno."""
    if not FAISS_AVAILABLE:
        raise RuntimeError("FAISS no está disponible para construir el índice HNSW.")
    dim = int(embeddings.shape[1])
    index = faiss.IndexHNSWFlat(dim, M, faiss.METRIC_INNER_PRODUCT)
    index.hnsw.efConstruction = ef_construction
    index.hnsw.efSearch = ef_search
    index.add(np.ascontiguousarray(embeddings.astype(np.float32)))
    return index


hnsw_index = build_hnsw_index(embeddings) if FAISS_AVAILABLE else None
print({
    "faiss_available": FAISS_AVAILABLE,
    "hnsw_index": None if hnsw_index is None else {
        "ntotal": int(hnsw_index.ntotal),
        "metric": "IP",
        "M": hnsw_index.hnsw.M,
        "efConstruction": hnsw_index.hnsw.efConstruction,
        "efSearch": hnsw_index.hnsw.efSearch,
    }
})


def search_dense_ann(query_text, top_k=10, filter_brand: Optional[str] = None):
    """Búsqueda semántica ANN con filtro por marca en la propia consulta."""
    q_vec = encode_query(query_text).reshape(1, -1)

    if hnsw_index is None:
        return search_dense(query_text, top_k=top_k, filter_brand=filter_brand)

    scores, indices = hnsw_index.search(np.ascontiguousarray(q_vec.astype(np.float32)), top_k + 40)
    scores = scores[0]
    indices = indices[0]

    results = []
    for rank, (score, idx) in enumerate(zip(scores, indices), start=1):
        if idx < 0:
            continue
        row = catalog_sample.iloc[int(idx)]
        if filter_brand is not None:
            if str(row.get("brand", "")).strip().lower() != str(filter_brand).strip().lower():
                continue
        results.append(
            SearchHit(
                record_id=str(row["record_id"]),
                product_id=str(row["product_id"]),
                title=str(row.get("title", "")),
                brand=str(row.get("brand", "")),
                score=float(score),
                score_kind="similarity",
                higher_is_better=True,
                rank=rank,
            )
        )
        if len(results) >= top_k:
            break

    return results


search_dense_ann("zapatillas para correr de marca nike", top_k=5, filter_brand="Nike")


## 7. Evaluación comparativa

Comparamos el baseline léxico y la búsqueda densa con las métricas solicitadas sobre desarrollo.

In [21]:
def evaluate_dense(queries, qrels, top_k=10):
    rows = []
    qrels_map = qrels_by_query(qrels)
    for _, query in queries.iterrows():
        qid = str(query["query_id"])
        results = search_dense(query["query_text"], top_k=top_k)
        ranked_ids = [r["product_id"] for r in results]
        relevant = qrels_map.get(qid, set())
        ranked_rels = [relevances_dev.loc[(relevances_dev["query_id"] == int(qid)) & (relevances_dev["product_id"] == pid), "relevance"].iloc[0] if pid in relevant else 0 for pid in ranked_ids]
        rows.append((int(qid), ndcg_at_k(ranked_rels, top_k), recall_at_k(ranked_ids, relevant, top_k), mrr_at_k(ranked_ids, relevant, top_k)))
    return pd.DataFrame(rows, columns=["query_id", "ndcg", "recall", "mrr"])

lexical_results = evaluate_search(search_lexical, queries_dev, relevances_dev)
dense_results = evaluate_dense(queries_dev, relevances_dev)
print("Lexical mean metrics:")
print(lexical_results[["ndcg", "recall", "mrr"]].mean().to_dict())
print("Dense mean metrics:")
print(dense_results[["ndcg", "recall", "mrr"]].mean().to_dict())

Lexical mean metrics:
{'ndcg': 0.8172634025173084, 'recall': 0.2015625, 'mrr': 0.8928571428571428}
Dense mean metrics:
{'ndcg': 0.7837052304160472, 'recall': 0.19687500000000002, 'mrr': 0.78125}


## 8. Búsqueda filtrada por marca

Aplicamos el filtro de marca como parte de la recuperación. Para aplicar el filtro en el momento de la busqueda.

In [22]:
for _, row in queries_filtered.iterrows():
    results = search_dense(row["query_text"], top_k=10, filter_brand=row["filter_value"])
    brands = {r["brand"] for r in results}
    print(row["workload_id"], row["filter_value"], brands)

FILTER-001 Einhell set()
FILTER-002 Apple set()
FILTER-003 NIKE set()
FILTER-004 SAMSUNG set()


## 9. Eventos de catálogo y mutaciones

Aplicamos los 24 eventos de `eventos_catalogo.csv` respetando el orden `sequence`.
El proceso es idempotente: repetirlo no debe cambiar el catálogo final.

In [23]:
current_catalog = catalog_sample.set_index("record_id", drop=False).copy()
upsert_fields = ["record_id", "product_id", "title", "brand", "color", "locale", "text", "catalog_version", "active"]
for _, event in events_catalog.sort_values("sequence").iterrows():
    record_id = str(event["record_id"])
    op = event["operation"].upper()
    if op == "UPSERT":
        fields = [f for f in upsert_fields if f in current_catalog.columns]
        current_catalog.loc[record_id, fields] = event.loc[fields].values
    elif op == "DELETE":
        current_catalog = current_catalog.drop(record_id, errors="ignore")
    else:
        raise ValueError(f"Operación desconocida: {op}")

print("Registros finales tras eventos:", current_catalog.shape)
current_catalog.head(3)

Registros finales tras eventos: (1500, 10)


,record_id,product_id,title,brand,color,locale,text,catalog_version,active,text_for_embedding
record_id,,,,,,,,,,
000bd6e8-a995-56d0-ba03-559885ccef39,000bd6e8-a995-56d0-ba03-559885ccef39,B0818K237B,Kanlin1986 Vestido Largo De Navidad para Mujer...,KanLin1986-Ropa,Negro,es,Kanlin1986 Vestido Largo De Navidad para Mujer...,1.0,True,Kanlin1986 Vestido Largo De Navidad para Mujer...
0037a9df-8492-508f-8167-c09624801216,0037a9df-8492-508f-8167-c09624801216,B086YX9RK5,IQOS Kit Iqos 3 Duo Blue Opk 1 200 g,IQOS,NaN,es,IQOS Kit Iqos 3 Duo Blue Opk 1 200 g. Marca: I...,1.0,True,IQOS Kit Iqos 3 Duo Blue Opk 1 200 g | Marca: ...
003a8544-5ab1-5c8e-8fa5-612989e4a7a8,003a8544-5ab1-5c8e-8fa5-612989e4a7a8,B07FRXCFJ1,ELINKUME Lámpara de pie regulable LED Lámpara ...,ELINKUME,Lámparas de Pie-espiral Estilo-regulable Led,es,ELINKUME Lámpara de pie regulable LED Lámpara ...,1.0,True,ELINKUME Lámpara de pie regulable LED Lámpara ...


In [24]:
current_catalog_second = current_catalog.copy()
upsert_fields = ["record_id", "product_id", "title", "brand", "color", "locale", "text", "catalog_version", "active"]
for _, event in events_catalog.sort_values("sequence").iterrows():
    record_id = str(event["record_id"])
    op = event["operation"].upper()
    if op == "UPSERT":
        fields = [f for f in upsert_fields if f in current_catalog_second.columns]
        current_catalog_second.loc[record_id, fields] = event.loc[fields].values
    elif op == "DELETE":
        current_catalog_second = current_catalog_second.drop(record_id, errors="ignore")

print("Idempotencia al repetir eventos:", current_catalog.equals(current_catalog_second))

Idempotencia al repetir eventos: True


## 10. Detección de duplicados

Utilizamos `altas_desarrollo.csv` para ajustar una regla reproducible basada en el score del candidato más cercano y el margen con respecto al segundo mejor candidato.

In [25]:
match_rows = []
for _, row in duplicate_dev.iterrows():
    text = build_product_text(row)
    candidates = search_dense(text, top_k=2)
    if not candidates:
        continue
    top_score = candidates[0]["score"]
    second_score = candidates[1]["score"] if len(candidates) > 1 else 0.0
    match_rows.append({
        "incoming_id": row["incoming_id"],
        "best_product_id": candidates[0]["product_id"],
        "top_score": top_score,
        "second_score": second_score,
        "margin": top_score - second_score,
        "is_duplicate": row["is_duplicate"],
    })
match_df = pd.DataFrame(match_rows)
match_df.head(5)

,incoming_id,best_product_id,top_score,second_score,margin,is_duplicate
0,DEV-DUP-001,B000G3T55M,0.992776,0.694619,0.298157,True
1,DEV-DUP-002,B07NV4L2W5,0.922767,0.693453,0.229314,True
2,DEV-DUP-003,B00BEFAR80,0.882893,0.462671,0.420222,True
3,DEV-DUP-004,B076HKFZ8N,0.860701,0.561318,0.299383,True
4,DEV-DUP-005,B07JYHSK27,0.949464,0.446662,0.502803,True


In [26]:
thresholds = np.linspace(0.40, 0.90, 11)
gaps = np.linspace(0.01, 0.20, 10)

best_metrics = None
best_score = -1
for t in thresholds:
    for g in gaps:
        preds = [(row["top_score"] >= t and row["margin"] >= g) for _, row in match_df.iterrows()]
        tp = sum(p and row.is_duplicate for p, row in zip(preds, match_df.itertuples()))
        fp = sum(p and not row.is_duplicate for p, row in zip(preds, match_df.itertuples()))
        fn = sum((not p) and row.is_duplicate for p, row in zip(preds, match_df.itertuples()))
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        if f1 > best_score:
            best_score = f1
            best_metrics = {
                "threshold": t,
                "gap": g,
                "precision": precision,
                "recall": recall,
                "f1": f1,
            }

best_metrics

{'threshold': np.float64(0.4),
 'gap': np.float64(0.052222222222222225),
 'precision': np.float64(1.0),
 'recall': np.float64(1.0),
 'f1': np.float64(1.0)}

## 11. Exportación de resultados

Generamos los artefactos en el directorio de /results:

- `resultados_busqueda.csv`
- `resultados_duplicados.csv`
- `metricas_desarrollo.json`

In [27]:
search_rows = []
for _, row in queries_eval.iterrows():
    results = search_dense(row["query_text"], top_k=10)
    for rank, item in enumerate(results, start=1):
        search_rows.append({
            "evaluation_id": row["evaluation_id"],
            "rank": rank,
            "product_id": item["product_id"],
            "score": item["score"],
        })

pd.DataFrame(search_rows).to_csv(RESULTS_DIR / "resultados_busqueda.csv", index=False)
print("resultados_busqueda.csv generado en results/")

resultados_busqueda.csv generado en results/


In [28]:
if (DATA_DIR / "altas_evaluacion.csv").exists():
    duplicate_eval = pd.read_csv(DATA_DIR / "altas_evaluacion.csv")
    duplicate_rows = []
    for _, row in duplicate_eval.iterrows():
        text = build_product_text(row)
        candidates = search_dense(text, top_k=2)
        top_score = candidates[0]["score"] if candidates else 0.0
        matched_id = candidates[0]["product_id"] if candidates else ""
        is_dup = top_score >= best_metrics["threshold"] and (top_score - (candidates[1]["score"] if len(candidates) > 1 else 0.0)) >= best_metrics["gap"]
        duplicate_rows.append({
            "incoming_id": row["incoming_id"],
            "predicted_duplicate": str(is_dup).lower(),
            "matched_product_id": matched_id if is_dup else "",
            "score": top_score,
        })
    pd.DataFrame(duplicate_rows).to_csv(RESULTS_DIR / "resultados_duplicados.csv", index=False)
    print("resultados_duplicados.csv generado en results/")
else:
    print("No se encontró data/altas_evaluacion.csv")

resultados_duplicados.csv generado en results/


In [29]:
start = time.perf_counter()
for _, row in queries_dev.iterrows():
    _ = search_dense(row["query_text"], top_k=10)
end = time.perf_counter()
latency_ms = ((end - start) / len(queries_dev)) * 1000

metrics = {
    "lexical": {
        "ndcg_at_10": float(lexical_results["ndcg"].mean()),
        "recall_at_10": float(lexical_results["recall"].mean()),
        "mrr_at_10": float(lexical_results["mrr"].mean()),
    },
    "dense": {
        "ndcg_at_10": float(dense_results["ndcg"].mean()),
        "recall_at_10": float(dense_results["recall"].mean()),
        "mrr_at_10": float(dense_results["mrr"].mean()),
    },
    "latency_p50_ms": latency_ms,
    "latency_p95_ms": latency_ms,
}
with open(RESULTS_DIR / "metricas_desarrollo.json", "w", encoding="utf-8") as fh:
    json.dump(metrics, fh, indent=2, ensure_ascii=False)
print("metricas_desarrollo.json generado en results/")
metrics

metricas_desarrollo.json generado en results/


{'lexical': {'ndcg_at_10': 0.8172634025173084,
  'recall_at_10': 0.2015625,
  'mrr_at_10': 0.8928571428571428},
 'dense': {'ndcg_at_10': 0.7837052304160472,
  'recall_at_10': 0.19687500000000002,
  'mrr_at_10': 0.78125},
 'latency_p50_ms': 9.673262499973134,
 'latency_p95_ms': 9.673262499973134}